# evaluation/01 — Seed Testing (Primary Results)

5-seed evaluation of all five primary ANTHEIA models.
These are the final reported results.

**Protocol:**
- Seeds: [42, 0, 1, 2, 3]
- Per seed: resample negatives at 3:1 ratio using `np.random.default_rng(seed)`
- Train/test split: 80/20 stratified, refitted per seed
- Metric: ROC-AUC and PR-AUC on held-out test set
- Report: mean ± std across 5 seeds

**Why 5 seeds?** The positive set is small (139 pairs). Variance across
seeds reflects sensitivity to which specific negative pairs are drawn.
5-seed mean ± std is the primary evidence for model comparison.

**Expected results (GBIF a_curves):**

| Model | ROC-AUC | PR-AUC |
|---|---|---|
| Spatial Baseline | 0.9571 ± 0.0375 | 0.9387 ± 0.0335 |
| ANTHEIA-Scalar | 0.9599 ± 0.0365 | 0.9403 ± 0.0339 |
| **ANTHEIA-4D** | **0.9602 ± 0.0232** | **0.9424 ± 0.0257** |
| ANTHEIA-15D | 0.9514 ± 0.0274 | 0.9374 ± 0.0245 |
| ANTHEIA-PMf | 0.9560 ± 0.0298 | 0.9357 ± 0.0338 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from pathlib import Path

BASE     = Path("/scratch/ariana.l")
OLD_S4   = BASE / "Stage 4 Link Prediction Model"
NEW_S4   = BASE / "New Stage 4 Link Prediction Model"
STAGE5   = BASE / "Stage 5 PPE Representation Study"

VF_PATH       = OLD_S4 / "stage4_Vf_phenofield.csv"
F_PATH        = OLD_S4 / "stage4_F_existence_phenofield.csv"
VP_PATH       = NEW_S4 / "stage4_Vp_corrected.csv"
P_PATH        = NEW_S4 / "stage4_P_existence_corrected.csv"
GLOBI_PATH    = OLD_S4 / "stage4_globi_conus_broad.csv"
F_CURVES_PATH = NEW_S4 / "f_curves_ppe.csv"
A_CURVES_PATH = NEW_S4 / "a_curves_corrected.csv"
VDELTA_4D     = STAGE5 / "stage5_Vdelta_ppe.csv"
VDELTA_15D    = STAGE5 / "stage5_Vdelta_15d.csv"
VF_PROB       = STAGE5 / "stage5_Vf_prob.csv"

SEEDS     = [42, 0, 1, 2, 3]
NEG_RATIO = 3

print("Paths OK")

In [ ]:
# Load all embeddings and curves
print("Loading embeddings...")
Vf_df   = pd.read_csv(VF_PATH,   index_col=0)
Vp_df   = pd.read_csv(VP_PATH,   index_col=0)
F_df    = pd.read_csv(F_PATH,    index_col=0)
P_df    = pd.read_csv(P_PATH,    index_col=0)
Vd4_df  = pd.read_csv(VDELTA_4D,  index_col=0)
Vd15_df = pd.read_csv(VDELTA_15D, index_col=0)
Vfp_df  = pd.read_csv(VF_PROB,    index_col=0)

f_curves_df = pd.read_csv(F_CURVES_PATH, index_col=0)
f_curves_df.columns = list(range(52))
a_curves_df = pd.read_csv(A_CURVES_PATH, index_col=0)
a_curves_df.columns = list(range(52))

common_bins = [b for b in F_df.columns if b in set(P_df.columns)]
F_common = F_df[common_bins].values
P_common = P_df[common_bins].values
fc_idx   = {sp: i for i, sp in enumerate(F_df.index)}
pc_idx   = {sp: i for i, sp in enumerate(P_df.index)}

def compute_N(plant, pollinator):
    return float(F_common[fc_idx[plant]] @ P_common[pc_idx[pollinator]])

print(f"All embeddings loaded. Common bins: {len(common_bins)}")

In [ ]:
# Shared pair universe
plants_all = (
    set(Vf_df.index) & set(f_curves_df.index) &
    set(Vd4_df.index) & set(Vd15_df.index) & set(Vfp_df.index)
)
polls_all = set(Vp_df.index) & set(a_curves_df.index)

globi = pd.read_csv(GLOBI_PATH)
globi = globi.rename(columns={
    "sourceTaxonName": "plant_species",
    "targetTaxonName": "pollinator_species"
})
globi = globi.dropna(subset=["plant_species", "pollinator_species"])
globi = globi[["plant_species", "pollinator_species"]].drop_duplicates()

globi_shared = globi[
    globi["plant_species"].isin(plants_all) &
    globi["pollinator_species"].isin(polls_all)
].reset_index(drop=True)

plant_list   = sorted(plants_all)
poll_list    = sorted(polls_all)
positive_set = set(zip(globi_shared["plant_species"], globi_shared["pollinator_species"]))
n_neg        = len(globi_shared) * NEG_RATIO

print(f"Positive pairs: {len(globi_shared)}")
print(f"Negatives per seed: {n_neg}")

In [ ]:
# 5-seed evaluation loop
model_names = [
    "Spatial Baseline",
    "ANTHEIA-Scalar",
    "ANTHEIA-4D",
    "ANTHEIA-15D",
    "ANTHEIA-PMf",
]
seed_results = {name: {"roc": [], "pr": []} for name in model_names}

for seed in SEEDS:
    rng_s = np.random.default_rng(seed)

    # Resample negatives
    negatives_s = []
    while len(negatives_s) < n_neg:
        pl_sample = rng_s.choice(plant_list, size=n_neg * 2)
        po_sample = rng_s.choice(poll_list,  size=n_neg * 2)
        for pl, po in zip(pl_sample, po_sample):
            if (pl, po) not in positive_set:
                negatives_s.append((pl, po))
            if len(negatives_s) >= n_neg:
                break

    neg_df_s = pd.DataFrame(negatives_s, columns=["plant_species", "pollinator_species"])
    pairs_s  = pd.concat([
        globi_shared.assign(label=1),
        neg_df_s.assign(label=0)
    ], ignore_index=True)
    y_s = pairs_s["label"].values

    # Assemble feature matrices
    rows = {name: [] for name in model_names}
    for _, row in pairs_s.iterrows():
        pl, po = row["plant_species"], row["pollinator_species"]
        vf    = Vf_df.loc[pl].values
        vp    = Vp_df.loc[po].values
        n     = compute_N(pl, po)
        f     = f_curves_df.loc[pl].values
        a     = a_curves_df.loc[po].values
        delta = np.minimum(f, a).sum()
        vd4   = Vd4_df.loc[pl].values
        vd15  = Vd15_df.loc[pl].values
        vfp   = Vfp_df.loc[pl].values

        rows["Spatial Baseline"].append(np.concatenate([vf, vp, [n]]))
        rows["ANTHEIA-Scalar"].append(np.concatenate([vf, vp, [n, delta]]))
        rows["ANTHEIA-4D"].append(np.concatenate([vf, vp, [n], vd4]))
        rows["ANTHEIA-15D"].append(np.concatenate([vf, vp, [n], vd15]))
        rows["ANTHEIA-PMf"].append(np.concatenate([vfp, vp, [n]]))

    X_matrices = {name: np.array(rows[name]) for name in model_names}

    idx = np.arange(len(pairs_s))
    tr_idx, te_idx = train_test_split(
        idx, test_size=0.2, random_state=seed, stratify=y_s
    )

    for name in model_names:
        X_m = X_matrices[name]
        clf = LogisticRegression(max_iter=1000, random_state=seed)
        clf.fit(X_m[tr_idx], y_s[tr_idx])
        y_prob = clf.predict_proba(X_m[te_idx])[:, 1]
        roc = roc_auc_score(y_s[te_idx], y_prob)
        pr  = average_precision_score(y_s[te_idx], y_prob)
        seed_results[name]["roc"].append(roc)
        seed_results[name]["pr"].append(pr)

    print(f"  Seed {seed} done.")

In [ ]:
# Print results
print(f"\n{'Model':<22} {'ROC-AUC':>20} {'PR-AUC':>20}")
print("-" * 65)
for name in model_names:
    roc_mean = np.mean(seed_results[name]["roc"])
    roc_std  = np.std(seed_results[name]["roc"])
    pr_mean  = np.mean(seed_results[name]["pr"])
    pr_std   = np.std(seed_results[name]["pr"])
    print(f"{name:<22} {roc_mean:.4f} ± {roc_std:.4f}   {pr_mean:.4f} ± {pr_std:.4f}")

In [ ]:
# Save results to CSV
rows_out = []
for name in model_names:
    rows_out.append({
        "Model"         : name,
        "ROC-AUC Mean"  : round(np.mean(seed_results[name]["roc"]), 4),
        "ROC-AUC Std"   : round(np.std(seed_results[name]["roc"]),  4),
        "PR-AUC Mean"   : round(np.mean(seed_results[name]["pr"]),  4),
        "PR-AUC Std"    : round(np.std(seed_results[name]["pr"]),   4),
    })

results_df = pd.DataFrame(rows_out)
results_df.to_csv(NEW_S4 / "corrected_model_results.csv", index=False)
print(f"Saved corrected_model_results.csv")
print(results_df.to_string(index=False))

In [ ]:
# Results table figure
best_roc = results_df["ROC-AUC Mean"].max()
best_pr  = results_df["PR-AUC Mean"].max()

cell_text = []
for _, row in results_df.iterrows():
    cell_text.append([
        row["Model"],
        f"{row['ROC-AUC Mean']:.4f} ± {row['ROC-AUC Std']:.4f}",
        f"{row['PR-AUC Mean']:.4f} ± {row['PR-AUC Std']:.4f}",
    ])

fig, ax = plt.subplots(figsize=(9, 2.6))
ax.axis("off")
table = ax.table(
    cellText=cell_text,
    colLabels=["Model", "ROC-AUC (mean ± std)", "PR-AUC (mean ± std)"],
    loc="center", cellLoc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

for j in range(3):
    table[0, j].set_facecolor("#2c2c2c")
    table[0, j].set_text_props(color="white", fontweight="bold")

for i, (_, row) in enumerate(results_df.iterrows(), start=1):
    bg = "#f5f5f5" if i % 2 == 0 else "white"
    table[i, 0].set_facecolor(bg)
    table[i, 1].set_facecolor("#e8f4e8" if row["ROC-AUC Mean"] == best_roc else bg)
    table[i, 2].set_facecolor("#e8f4e8" if row["PR-AUC Mean"]  == best_pr  else bg)
    table[i, 1].set_text_props(fontweight="bold" if row["ROC-AUC Mean"] == best_roc else "normal")
    table[i, 2].set_text_props(fontweight="bold" if row["PR-AUC Mean"]  == best_pr  else "normal")

plt.title("ANTHEIA Results (5-seed mean ± std, GBIF a_curves)",
          fontsize=12, fontweight="bold", pad=12)
plt.savefig(NEW_S4 / "corrected_model_results.png",
            dpi=300, bbox_inches="tight", facecolor="white")
print("Saved corrected_model_results.png")